# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heyzara124-hub/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**My lane: CTR / Engagement Opportunity Scoring.**

**Task type: Ranking / Scoring** (not classification, not clustering).

I'm not sorting pages into two fixed buckets like "good" / "bad" (that would be classification),
and I'm not looking for groups of similar pages (that would be clustering). Instead, I'm giving
**every visible page a score** — "how much is this page under-performing for its position" —
and then **ranking** pages from the biggest opportunity to the smallest.

Why this fits: a content reviewer only has time to check a handful of pages a week. A ranked
list with a score tells them exactly which pages to open first. A plain yes/no label would
throw away the "how much" information that the ranking actually needs.


In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("Total pages in starter dataset:", len(df))

# Why "ranking", not a single yes/no rule: pages sit at very different positions,
# and position changes what a "normal" CTR looks like.
print(df["position_tier"].value_counts())


Total pages in starter dataset: 30000
position_tier
page_1      11814
striking     7304
page_3_5     7242
top_3        2321
deep         1319
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proxy target: `ctr_gap`** — a page's actual `ctr` minus the median `ctr` of other visible
pages that share its `position_tier`.

- This is a **proxy**, not a strict label like the starter pipeline's `is_declining_label`.
- It is built entirely from **observed measurements** (real clicks, real impressions) — not
  from someone's hand-picked rule — so it passes the "target must be observed" check from the
  framing skill.
- A negative `ctr_gap` means the page earns fewer clicks than similar pages *at the same
  position* → that is the "opportunity."
- I compare pages **only within the same position tier**, because a page-1 result naturally
  gets far more clicks than a page-15 result. Comparing raw CTR across tiers would be comparing
  apples to oranges — the position adjustment is the whole point of this lane.


In [2]:
# Only compare pages that actually have enough traffic to trust ("volume floor")
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()
print("Visible, high-enough-volume pages:", len(visible), "of", len(df))

# Expected CTR = median CTR of other pages in the same position tier
tier_median_ctr = visible.groupby("position_tier")["ctr"].median()
print("\nMedian CTR by position tier (this is our 'expected' CTR):")
print(tier_median_ctr)

visible["tier_median_ctr"] = visible["position_tier"].map(tier_median_ctr)
visible["ctr_gap"] = visible["ctr"] - visible["tier_median_ctr"]

visible[["content_id", "position_tier", "ctr", "tier_median_ctr", "ctr_gap"]].sort_values("ctr_gap").head()


Visible, high-enough-volume pages: 12023 of 30000

Median CTR by position tier (this is our 'expected' CTR):
position_tier
page_1      0.240
page_3_5    0.155
striking    0.170
top_3       0.200
Name: ctr, dtype: float64


,content_id,position_tier,ctr,tier_median_ctr,ctr_gap
15334,content_bb6c9ae71383,page_1,0.0,0.24,-0.24
461,content_29edd4a67cf6,page_1,0.0,0.24,-0.24
6648,content_10d038b1695e,page_1,0.0,0.24,-0.24
23745,content_c59a181ee3c0,page_1,0.0,0.24,-0.24
18715,content_96472c1f2070,page_1,0.0,0.24,-0.24


## 3. Success metric

*One metric you can defend. What number means "good"?*

**Success metric (for this framing stage): estimated extra clicks recovered if the top-K
flagged pages reached their tier's median CTR.**

This starter CSV is a single 90-day snapshot — it has no "did the CTR actually improve later"
outcome to check against yet (that lives in the bigger warehouse release, weeks 3+). So right
now I use a metric I can compute and defend **today**:

> If we picked the top 50 pages with the worst `ctr_gap` and got each one just up to its tier's
> median CTR, how many more clicks would that be over 90 days?

Later, once outcome data is available, the real test becomes **precision@50**: of the top 50
flagged pages, how many actually improved after someone acted on them. For now I only claim
"decision-support," not a proven outcome.


In [3]:
top50 = visible.sort_values("ctr_gap").head(50).copy()
top50["potential_extra_clicks"] = (top50["tier_median_ctr"] - top50["ctr"]) / 100 * top50["impressions_90d"]

print("Success metric on this starter slice:")
print("  Pages flagged (top 50 worst ctr_gap)")
print("  Estimated extra clicks/90d if all reached tier median:", round(top50["potential_extra_clicks"].sum()))
print("  Average ctr_gap among flagged pages (percentage points):", round(top50["ctr_gap"].mean(), 2))
print("  Distinct clients represented in the top 50:", top50["client_id"].nunique())


Success metric on this starter slice:
  Pages flagged (top 50 worst ctr_gap)
  Estimated extra clicks/90d if all reached tier median: 181
  Average ctr_gap among flagged pages (percentage points): -0.24
  Distinct clients represented in the top 50: 12


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page (`content_id`)** — already carrying its impressions, its real
CTR, the "expected" CTR for its position tier, and the gap between the two. That gap is what
gets ranked.


In [4]:
unit_of_analysis = top50[[
    "content_id", "client_id", "position_tier",
    "impressions_90d", "ctr", "tier_median_ctr", "ctr_gap",
    "sessions_90d", "engagement_rate", "potential_extra_clicks"
]].reset_index(drop=True)

print("One row = one content page. Top opportunities first:")
unit_of_analysis.head(10)


One row = one content page. Top opportunities first:


,content_id,client_id,position_tier,impressions_90d,ctr,tier_median_ctr,ctr_gap,sessions_90d,engagement_rate,potential_extra_clicks
0,content_bb6c9ae71383,client_6208ef0f77,page_1,1475,0.0,0.24,-0.24,7,0.00,3.5400
1,content_29edd4a67cf6,client_19581e27de,page_1,1268,0.0,0.24,-0.24,3,33.33,3.0432
2,content_10d038b1695e,client_19581e27de,page_1,599,0.0,0.24,-0.24,21,0.00,1.4376
3,content_c59a181ee3c0,client_4e07408562,page_1,669,0.0,0.24,-0.24,1,0.00,1.6056
4,content_96472c1f2070,client_19581e27de,page_1,714,0.0,0.24,-0.24,2,0.00,1.7136
5,content_e8ab88172458,client_a88a7902cb,page_1,669,0.0,0.24,-0.24,6,0.00,1.6056
6,content_4b1303a5affe,client_7f2253d7e2,page_1,1341,0.0,0.24,-0.24,1,0.00,3.2184
7,content_0895470266fd,client_6208ef0f77,page_1,709,0.0,0.24,-0.24,2,0.00,1.7016
8,content_cbae09e4ff9b,client_349c41201b,page_1,792,0.0,0.24,-0.24,8,0.00,1.9008
9,content_0ad759bc5d3d,client_19581e27de,page_1,3409,0.0,0.24,-0.24,10,0.00,8.1816


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single fixed rule like "flag anything under 1% CTR" breaks immediately on this data, because
"normal" CTR depends on several things at once, not just one number:

- **Position** — page-1 pages average ~0.24% CTR here, `striking`-tier pages average ~0.17%.
  A flat threshold either over-flags deep pages or under-flags page-1 pages.
- **Search intent and content type** — a transactional query behaves differently from an
  informational one; a single cutoff can't tell them apart.
- **Volume** — a page with 50 impressions and a page with 50,000 impressions need very
  different confidence before you trust their CTR number.

These signals interact rather than stacking cleanly, which is exactly when a learned model
earns its place over hand-written if/else rules. This repo already shows that pattern on a
neighboring lane: the starter pipeline's own report (`outputs/model_report.md`) compares a
hand-written baseline rule against trained models on the same 30k rows, predicting whether a
page is declining. The rule alone reached ROC AUC 0.627 / Precision@50 0.240, while a random
forest reached ROC AUC 0.750 / Precision@50 0.740 — meaning the top 50 pages it flagged were
right about 3x more often than the fixed rule's top 50. I'd expect a similar gap once I move
from "median-per-tier" to a real trained model that also weighs volume, intent, and content
type together instead of one at a time.


In [5]:
# Read the real numbers straight from this repo's own model report, instead of typing them by hand
with open("../../outputs/model_report.md") as f:
    report = f.read()

start = report.index("## Model Comparison")
end = report.index("## Final Queue")
print(report[start:end].strip())


## Model Comparison

Best model: `random_forest` selected by `precision_at_50`.

| Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
| logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.